# IDX Stock Scoring Model

## Overview
This notebook ranks Indonesia Stock Exchange (IDX) equities using a transparent multi-factor scoring framework. It combines market data and company metrics into a single **Total Score (0-100)** for screening and prioritization.

## Factor Framework and Weights
- Quality: 25%
- Growth: 25%
- Income: 20%
- Valuation: 15%
- Risk: 10%
- Momentum: 5%

Scoring formula:

`Total Score = sum(Factor Score x Weight)`

## Data Sources
- IDX universe and metadata: `pasardana.id`
- Market and fundamentals data: Yahoo Finance (`yfinance`)

## Output
The final output is a ranked stock table (`ranked_stocks`) with both aggregate and per-factor scores, plus optional sector-level ranking views.

## Notes
- This model is for research and educational use.
- Scores are data-dependent and can change as source data updates.


In [14]:
import numpy as np
import pandas as pd
import requests
import yfinance as yf
from time import sleep

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)


## 1) Build the IDX Coverage Universe and Raw Metrics

This section pulls the stock universe, maps IDX tickers to Yahoo symbols (`.JK`), retrieves core market and fundamentals fields, and merges both sources into one working dataset.


In [15]:
# Load IDX stock list
url = 'https://pasardana.id/api/StockSearchResult/GetAll'
response = requests.get(url, timeout=30)
response.raise_for_status()
stock_list = response.json()

df_metrics_indo = pd.DataFrame(stock_list)
df_metrics_indo['Ticker'] = df_metrics_indo['Code'].astype(str) + '.JK'

tickers = df_metrics_indo['Ticker'].dropna().unique().tolist()
len(tickers)

972

In [16]:
def get_stock_metrics(ticker_list, delay=0.35):
    data = []
    for ticker in ticker_list:
        sleep(delay)
        stock = yf.Ticker(ticker)
        try:
            info = stock.info
        except Exception:
            info = {}

        float_shares = info.get('floatShares')
        shares_outstanding = info.get('sharesOutstanding')
        free_float_ratio = (
            float_shares / shares_outstanding
            if isinstance(float_shares, (int, float))
            and isinstance(shares_outstanding, (int, float))
            and shares_outstanding
            else None
        )

        data.append({
            'Ticker': ticker,
            'Name': info.get('shortName'),
            'Sector': info.get('sector'),
            'Current Price': info.get('currentPrice'),
            'Market Cap': info.get('marketCap'),
            'PE Ratio': info.get('trailingPE'),
            'Dividend Yield': info.get('dividendYield'),
            'Debt to Equity': info.get('debtToEquity'),
            'Revenue Growth': info.get('revenueGrowth'),
            'EPS Growth': info.get('earningsQuarterlyGrowth'),
            'ROA': info.get('returnOnAssets'),
            'Operating Margin': info.get('operatingMargins'),
            'Net Margin': info.get('profitMargins'),
            'EV/EBITDA': info.get('enterpriseToEbitda'),
            'Beta': info.get('beta'),
            'Payout Ratio': info.get('payoutRatio'),
            'Current Ratio': info.get('currentRatio'),
            'Quick Ratio': info.get('quickRatio'),
            'Free Float Ratio': free_float_ratio,
            'Currency': info.get('currency'),
        })

    return pd.DataFrame(data)


In [17]:
df_yf = get_stock_metrics(tickers)
df_yf = df_yf[df_yf['Current Price'].notna()].copy()
merged_df = pd.merge(df_yf, df_metrics_indo, on='Ticker', how='left')
merged_df.head()


,Ticker,Name_x,Sector,Current Price,Market Cap,PE Ratio,Dividend Yield,Debt to Equity,Revenue Growth,EPS Growth,ROA,Operating Margin,Net Margin,EV/EBITDA,Beta,Payout Ratio,Current Ratio,Quick Ratio,Free Float Ratio,Currency,Id,Name_y,Code,StockSubSectorId,SubSectorName,StockSectorId,SectorName,NewSubIndustryId,NewSubIndustryName,NewIndustryId,NewIndustryName,NewSubSectorId,NewSubSectorName,NewSectorId,NewSectorName,Last,PrevClosingPrice,AdjustedClosingPrice,AdjustedOpenPrice,AdjustedHighPrice,AdjustedLowPrice,Volume,Frequency,Value,OneDay,OneWeek,OneMonth,ThreeMonth,SixMonth,OneYear,ThreeYear,FiveYear,TenYear,Mtd,Ytd,Per,Pbr,Capitalization,BetaOneYear,StdevOneYear,PerAnnualized,PsrAnnualized,PcfrAnnualized,AdjustedAnnualHighPrice,AdjustedAnnualLowPrice,LastDate,LastUpdate,Roe
0,AALI.JK,Astra Agro Lestari Tbk.,Consumer Defensive,8400.0,1.616738e+13,10.989011,3.65,NaN,0.182,0.159,0.04963,0.10872,0.05136,3.817,0.215,0.4015,4.331,1.997,0.203150,IDR,1,Astra Agro Lestari Tbk.,AALI,2.0,Plantation,1.0,AGRICULTURE,47.0,Perkebunan & Tanaman Pangan,22.0,Produk Makanan Pertanian,8.0,Makanan & Minuman,4.0,Barang Konsumen Primer,8400.0,8400.0,8400.0,8425.0,8450.0,8300.0,1100700.0,1312.0,9.222640e+09,0.000000,0.040248,0.166667,0.127517,0.046729,0.493333,0.066667,-0.142857,-0.450729,0.112583,0.138983,10.98541,0.66925,1.616738e+13,0.468959,0.257396,10.98541,0.56421,3.83970,9000.0,5575.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.060921
1,ABBA.JK,Mahaka Media Tbk.,Communication Services,41.0,1.613716e+11,None,NaN,NaN,-0.276,NaN,-0.03863,-0.11429,-0.12818,-32.456,-0.205,0.0000,0.497,0.340,0.385570,IDR,2,Mahaka Media Tbk.,ABBA,29.0,"Advertising, Printing, dan Media",8.0,"TRADE, SERVICES, & INVESTMENT",73.0,Penerbitan,34.0,Media,16.0,Media & Hiburan,5.0,Barang Konsumen Non-Primer,41.0,41.0,41.0,41.0,41.0,41.0,300100.0,65.0,1.230410e+07,0.000000,0.000000,0.000000,-0.280702,-0.316667,0.863636,-0.605769,-0.859589,-0.180000,-0.023810,-0.254545,-6.82568,-1.61633,1.613716e+11,1.242523,0.768610,-6.82568,0.87491,57.76091,97.0,22.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.236801
2,ABDA.JK,Asuransi Bina Dana Arta Tbk.,Financial Services,2590.0,1.607889e+12,16.092953,NaN,NaN,0.086,33.934,0.02657,0.13537,0.10244,-4.804,-0.134,0.0000,2.467,2.342,0.052299,IDR,3,Asuransi Bina Dana Arta Tbk.,ABDA,23.0,Insurance,7.0,FINANCE,97.0,Asuransi Umum,48.0,Asuransi,23.0,Asuransi,7.0,Keuangan,2590.0,2600.0,2590.0,2600.0,2600.0,2590.0,2900.0,3.0,7.513000e+06,-0.003846,0.052846,-0.040741,-0.307487,-0.044280,-0.238235,-0.600000,-0.632624,-0.674214,-0.003846,-0.282548,16.11561,0.90332,1.607889e+12,0.065182,0.500775,16.11561,1.49651,159.15802,4080.0,2230.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.056052
3,ABMM.JK,ABM Investama Tbk.,Energy,3050.0,8.397153e+12,6.933551,5.03,110.073,-0.163,0.030,0.01645,0.13707,0.06802,49549.043,-0.767,0.3567,1.158,1.009,0.099740,IDR,4,ABM Investama Tbk.,ABMM,7.0,Coal Mining,2.0,MINING,3.0,Produksi Batu Bara,2.0,Batu Bara,1.0,"Minyak, Gas & Batu Bara",1.0,Energi,3050.0,3040.0,3050.0,3040.0,3060.0,3010.0,490800.0,472.0,1.491492e+09,0.003289,-0.016129,0.066434,0.030405,0.077739,-0.061538,-0.072948,2.885350,0.115174,-0.019293,0.097122,7.08571,0.56833,8.397153e+12,0.464090,0.234937,7.08571,0.48197,1.69011,3430.0,2550.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.080207
4,ACES.JK,Aspirasi Hidup Indonesia Tbk.,Consumer Cyclical,386.0,6.608470e+12,9.910141,9.10,15.696,-0.014,-0.097,0.08118,0.12093,0.09195,4.992,0.190,0.8694,9.360,3.129,0.399900,IDR,6,Aspirasi Hidup Indonesia Tbk.,ACES,27.0,Retail Trade,8.0,"TRADE, SERVICES, & INVESTMENT",80.0,Ritel Barang Rumah Tangga,39.0,Ritel Khusus,17.0,Perdagangan Ritel,5.0,Barang Konsumen Non-Primer,386.0,372.0,386.0,372.0,386.0,372.0,41834800.0,4853.0,1.595020e+10,0.037634,0.078212,0.010471,-0.063107,-0.098131,-0.192469,-0.122727,-0.740067,-0.545882,0.021164,-0.058537,9.88217,1.00170,6.608470e+12,0.583184,0.324902,9.88217,0.76499,4.01285,610.0,348.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.101364


In [18]:
merged_df.to_csv('backup_data.csv', index=False)

## 2) Clean Data and Prepare Scoring Inputs

This section standardizes numeric fields, handles missing values, and applies helper logic so all factors can be scored consistently on comparable inputs.


In [19]:
# read idx_stock_metrics.csv
df_metrics = pd.read_csv('backup_data.csv')
df_metrics.head()

,Ticker,Name_x,Sector,Current Price,Market Cap,PE Ratio,Dividend Yield,Debt to Equity,Revenue Growth,EPS Growth,ROA,Operating Margin,Net Margin,EV/EBITDA,Beta,Payout Ratio,Current Ratio,Quick Ratio,Free Float Ratio,Currency,Id,Name_y,Code,StockSubSectorId,SubSectorName,StockSectorId,SectorName,NewSubIndustryId,NewSubIndustryName,NewIndustryId,NewIndustryName,NewSubSectorId,NewSubSectorName,NewSectorId,NewSectorName,Last,PrevClosingPrice,AdjustedClosingPrice,AdjustedOpenPrice,AdjustedHighPrice,AdjustedLowPrice,Volume,Frequency,Value,OneDay,OneWeek,OneMonth,ThreeMonth,SixMonth,OneYear,ThreeYear,FiveYear,TenYear,Mtd,Ytd,Per,Pbr,Capitalization,BetaOneYear,StdevOneYear,PerAnnualized,PsrAnnualized,PcfrAnnualized,AdjustedAnnualHighPrice,AdjustedAnnualLowPrice,LastDate,LastUpdate,Roe
0,AALI.JK,Astra Agro Lestari Tbk.,Consumer Defensive,8400.0,1.616738e+13,10.989011,3.65,NaN,0.182,0.159,0.04963,0.10872,0.05136,3.817,0.215,0.4015,4.331,1.997,0.203150,IDR,1,Astra Agro Lestari Tbk.,AALI,2.0,Plantation,1.0,AGRICULTURE,47.0,Perkebunan & Tanaman Pangan,22.0,Produk Makanan Pertanian,8.0,Makanan & Minuman,4.0,Barang Konsumen Primer,8400.0,8400.0,8400.0,8425.0,8450.0,8300.0,1100700.0,1312.0,9.222640e+09,0.000000,0.040248,0.166667,0.127517,0.046729,0.493333,0.066667,-0.142857,-0.450729,0.112583,0.138983,10.98541,0.66925,1.616738e+13,0.468959,0.257396,10.98541,0.56421,3.83970,9000.0,5575.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.060921
1,ABBA.JK,Mahaka Media Tbk.,Communication Services,41.0,1.613716e+11,NaN,NaN,NaN,-0.276,NaN,-0.03863,-0.11429,-0.12818,-32.456,-0.205,0.0000,0.497,0.340,0.385570,IDR,2,Mahaka Media Tbk.,ABBA,29.0,"Advertising, Printing, dan Media",8.0,"TRADE, SERVICES, & INVESTMENT",73.0,Penerbitan,34.0,Media,16.0,Media & Hiburan,5.0,Barang Konsumen Non-Primer,41.0,41.0,41.0,41.0,41.0,41.0,300100.0,65.0,1.230410e+07,0.000000,0.000000,0.000000,-0.280702,-0.316667,0.863636,-0.605769,-0.859589,-0.180000,-0.023810,-0.254545,-6.82568,-1.61633,1.613716e+11,1.242523,0.768610,-6.82568,0.87491,57.76091,97.0,22.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.236801
2,ABDA.JK,Asuransi Bina Dana Arta Tbk.,Financial Services,2590.0,1.607889e+12,16.092953,NaN,NaN,0.086,33.934,0.02657,0.13537,0.10244,-4.804,-0.134,0.0000,2.467,2.342,0.052299,IDR,3,Asuransi Bina Dana Arta Tbk.,ABDA,23.0,Insurance,7.0,FINANCE,97.0,Asuransi Umum,48.0,Asuransi,23.0,Asuransi,7.0,Keuangan,2590.0,2600.0,2590.0,2600.0,2600.0,2590.0,2900.0,3.0,7.513000e+06,-0.003846,0.052846,-0.040741,-0.307487,-0.044280,-0.238235,-0.600000,-0.632624,-0.674214,-0.003846,-0.282548,16.11561,0.90332,1.607889e+12,0.065182,0.500775,16.11561,1.49651,159.15802,4080.0,2230.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.056052
3,ABMM.JK,ABM Investama Tbk.,Energy,3050.0,8.397153e+12,6.933551,5.03,110.073,-0.163,0.030,0.01645,0.13707,0.06802,49549.043,-0.767,0.3567,1.158,1.009,0.099740,IDR,4,ABM Investama Tbk.,ABMM,7.0,Coal Mining,2.0,MINING,3.0,Produksi Batu Bara,2.0,Batu Bara,1.0,"Minyak, Gas & Batu Bara",1.0,Energi,3050.0,3040.0,3050.0,3040.0,3060.0,3010.0,490800.0,472.0,1.491492e+09,0.003289,-0.016129,0.066434,0.030405,0.077739,-0.061538,-0.072948,2.885350,0.115174,-0.019293,0.097122,7.08571,0.56833,8.397153e+12,0.464090,0.234937,7.08571,0.48197,1.69011,3430.0,2550.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.080207
4,ACES.JK,Aspirasi Hidup Indonesia Tbk.,Consumer Cyclical,386.0,6.608470e+12,9.910141,9.10,15.696,-0.014,-0.097,0.08118,0.12093,0.09195,4.992,0.190,0.8694,9.360,3.129,0.399900,IDR,6,Aspirasi Hidup Indonesia Tbk.,ACES,27.0,Retail Trade,8.0,"TRADE, SERVICES, & INVESTMENT",80.0,Ritel Barang Rumah Tangga,39.0,Ritel Khusus,17.0,Perdagangan Ritel,5.0,Barang Konsumen Non-Primer,386.0,372.0,386.0,372.0,386.0,372.0,41834800.0,4853.0,1.595020e+10,0.037634,0.078212,0.010471,-0.063107,-0.098131,-0.192469,-0.122727,-0.740067,-0.545882,0.021164,-0.058537,9.88217,1.00170,6.608470e+12,0.583184,0.324902,9.88217,0.76499,4.01285,610.0,348.0,2026-04-21T00:00:00,2026-04-21T00:00:00,0.101364


In [20]:
score_df = merged_df.copy()

num_cols = [
    'PE Ratio', 'Pbr', 'Dividend Yield', 'Roe', 'ROA',
    'Debt to Equity', 'Revenue Growth', 'EPS Growth', 'Operating Margin',
    'Net Margin', 'EV/EBITDA', 'Beta',
    'OneYear', 'SixMonth', 'ThreeMonth', 'Payout Ratio', 'Current Ratio',
    'Quick Ratio', 'Market Cap', 'Frequency', 'Free Float Ratio'
]

for c in num_cols:
    if c in score_df.columns:
        score_df[c] = pd.to_numeric(score_df[c], errors='coerce')

def winsorize_series(s, q_low=0.05, q_high=0.95):
    s = s.copy()
    lo = s.quantile(q_low)
    hi = s.quantile(q_high)
    return s.clip(lower=lo, upper=hi)

def pct_rank_score(s, higher_is_better=True):
    x = winsorize_series(pd.to_numeric(s, errors='coerce'))
    rank = x.rank(pct=True, method='average')
    if not higher_is_better:
        rank = 1 - rank
    return (rank * 100).fillna(50)


## 3) Apply Eligibility Filters (Quality and Liquidity)

This section removes names that do not meet minimum investability thresholds before ranking, helping reduce low-quality or illiquid candidates.


In [21]:
hard_filtered = score_df.copy()

hard_filtered = hard_filtered[hard_filtered['Current Price'].notna()]
hard_filtered = hard_filtered[hard_filtered['Roe'].fillna(-999) >= 0.14]  # >= 14%
hard_filtered = hard_filtered[hard_filtered['Debt to Equity'].fillna(9999) < 300]
hard_filtered = hard_filtered[hard_filtered['PE Ratio'].fillna(9999) < 20] # between 5 and 20
hard_filtered = hard_filtered[hard_filtered['PE Ratio'].fillna(9999) > 5] # between 5 and 20
hard_filtered = hard_filtered[hard_filtered['Frequency'].fillna(-999) > 100] # above 100
hard_filtered = hard_filtered[hard_filtered['Dividend Yield'].fillna(-999) > 5.11] # dividend yield above 5.11%
hard_filtered = hard_filtered[hard_filtered['Free Float Ratio'].fillna(-999) >= 0.10] # Free Float Ratio above 10%

hard_filtered.shape


(21, 68)

## 4) Calculate Factor Scores (0-100)

This section computes normalized scores for each factor pillar (Quality, Valuation, Growth, Momentum, Risk, and Income) to create a balanced multi-factor profile per stock.


In [22]:
df = hard_filtered.copy()

# Quality
quality_components = pd.DataFrame({
    'ROE': pct_rank_score(df['Roe'], higher_is_better=True),
    'ROA': pct_rank_score(df['ROA'], higher_is_better=True),
    'Operating Margin': pct_rank_score(df['Operating Margin'], higher_is_better=True),
    'Net Margin': pct_rank_score(df['Net Margin'], higher_is_better=True),
})
df['Quality Score'] = quality_components.mean(axis=1)

# Valuation
valuation_components = pd.DataFrame({
    'PE Ratio': pct_rank_score(df['PE Ratio'], higher_is_better=False),
    'PB Effective': pct_rank_score(df['Pbr'], higher_is_better=False),
    'EV/EBITDA': pct_rank_score(df['EV/EBITDA'], higher_is_better=False),
})
df['Valuation Score'] = valuation_components.mean(axis=1)

# Growth
growth_components = pd.DataFrame({
    'Revenue Growth': pct_rank_score(df['Revenue Growth'], higher_is_better=True),
    'EPS Growth': pct_rank_score(df['EPS Growth'], higher_is_better=True),
})
df['Growth Score'] = growth_components.mean(axis=1)

# Momentum
momentum_components = pd.DataFrame({
    'ThreeMonth': pct_rank_score(df['ThreeMonth'], higher_is_better=True),
    'SixMonth': pct_rank_score(df['SixMonth'], higher_is_better=True),
    '1Y Return Effective': pct_rank_score(df['OneYear'], higher_is_better=True),
})
df['Momentum Score'] = momentum_components.mean(axis=1)

# Risk - lower beta/leverage and higher trading frequency are better
risk_components = pd.DataFrame({
    'Debt to Equity': pct_rank_score(df['Debt to Equity'], higher_is_better=False),
    'Beta': pct_rank_score(df['Beta'], higher_is_better=False),
    'Frequency': pct_rank_score(df['Frequency'], higher_is_better=True),
})
df['Risk Score'] = risk_components.mean(axis=1)

# Income
income_components = pd.DataFrame({
    'Dividend Yield': pct_rank_score(df['Dividend Yield'], higher_is_better=True),
    # Reward sustainable payout: closer to 40% is preferred
    'Payout Sustainability': pct_rank_score((df['Payout Ratio'] - 0.40).abs(), higher_is_better=False),
})
df['Income Score'] = income_components.mean(axis=1)

df[['Quality Score', 'Valuation Score', 'Growth Score', 'Momentum Score', 'Risk Score', 'Income Score']].head()


,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score
18,11.309524,36.984127,72.5,77.777778,70.555556,31.785714
36,48.809524,63.174603,30.0,29.365079,51.428571,58.095238
72,57.142857,71.428571,30.0,63.492063,23.095238,68.452381
192,13.690476,55.634921,87.5,50.793651,28.809524,67.857143
233,29.761905,75.714286,90.0,53.968254,35.396825,46.071429


## 5) Compute Weighted Total Score and Rank Stocks

This section combines factor scores using predefined weights, generates the final ranking table, and surfaces the highest-scoring opportunities.


In [23]:
weights = {
    'Quality Score': 0.25,
    'Growth Score': 0.25,
    'Income Score': 0.20,
    'Valuation Score': 0.15,
    'Risk Score': 0.10,
    'Momentum Score': 0.05
}

df['Total Score'] = sum(df[k] * w for k, w in weights.items())
df['Rank'] = df['Total Score'].rank(ascending=False, method='dense').astype(int)

rank_cols = [
    'Rank', 'Ticker', 'Sector', 'NewSubSectorName', 'NewSubIndustryName',
    'Total Score', 'Quality Score', 'Valuation Score', 'Growth Score',
    'Momentum Score', 'Risk Score', 'Income Score',
    'Current Price', 'Market Cap', 'PE Ratio', 'Pbr', 'Dividend Yield',
    'Roe', 'Debt to Equity', 'Revenue Growth', 'EPS Growth',
    'ThreeMonth', 'SixMonth', 'OneYear', 'Beta', 'Frequency', 'Payout Ratio', 'Free Float Ratio'
]

ranked_stocks = df[rank_cols].sort_values(['Total Score', 'Market Cap'], ascending=[False, False]).reset_index(drop=True)
ranked_stocks.head(30)


,Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Current Price,Market Cap,PE Ratio,Pbr,Dividend Yield,Roe,Debt to Equity,Revenue Growth,EPS Growth,ThreeMonth,SixMonth,OneYear,Beta,Frequency,Payout Ratio,Free Float Ratio
0,1,AADI.JK,Energy,"Minyak, Gas & Batu Bara",Produksi Batu Bara,64.569444,51.190476,56.349206,57.50,95.238095,56.349206,92.738095,10875.0,8.468244e+13,6.495407,1.38261,9.83,0.208290,22.636,0.025,0.270,0.464646,0.355140,0.611111,NaN,6460.0,0.3312,0.372190
1,2,MARK.JK,Healthcare,Barang Perindustrian,Mesin & Komponen Perindustrian,63.897817,93.452381,34.126984,71.25,49.206349,28.809524,61.309524,800.0,3.040000e+12,10.706638,3.36792,11.25,0.314501,7.307,0.145,0.219,-0.036145,0.194030,-0.064327,0.295,357.0,0.9371,0.197320
2,3,TOTL.JK,Industrials,Konstruksi Bangunan,Konstruksi Bangunan,57.208333,31.547619,42.142857,90.00,93.650794,54.365079,51.904762,1220.0,4.160200e+12,10.039500,3.12802,6.70,0.311516,0.367,0.286,0.399,0.161905,0.378531,1.033333,0.241,2014.0,0.6173,0.302580
3,4,JPFA.JK,Consumer Defensive,Makanan & Minuman,"Ikan, Daging & Produk Unggas",56.750000,29.761905,75.714286,90.00,53.968254,35.396825,46.071429,2650.0,3.081332e+13,7.702369,1.55230,5.20,0.200010,59.254,0.213,0.725,-0.095563,0.108787,0.391076,0.224,3765.0,0.2035,0.583353
4,5,MSTI.JK,Technology,Perangkat Lunak & Jasa TI,Jasa & Konsultan TI,56.365079,42.857143,68.095238,60.00,36.507937,60.158730,62.976190,1425.0,4.473668e+12,8.145650,1.95233,9.44,0.239622,4.343,0.151,0.094,0.010638,-0.106583,0.010638,-0.374,504.0,0.7662,0.149970
5,6,KMDS.JK,Consumer Defensive,Perdagangan Ritel Barang Primer,Ritel & Distributor Makanan,56.359127,52.380952,71.587302,77.50,28.571429,48.174603,34.523810,595.0,4.760000e+11,6.526198,1.84279,6.55,0.216302,0.509,0.320,0.169,-0.070312,-0.077519,-0.016529,0.217,452.0,NaN,0.209380
6,7,SMSM.JK,Consumer Cyclical,Otomotif & Komponen Otomotif,Suku Cadang Otomotif,55.746032,72.619048,53.412698,40.00,50.793651,53.253968,58.571429,1785.0,1.027924e+13,9.157603,2.38675,7.84,0.261147,4.017,0.054,0.010,0.017094,-0.011080,0.031792,0.131,904.0,0.7179,0.414350
7,8,IPCC.JK,Industrials,Infrastruktur Transportasi,Operator Pelabuhan,53.136905,58.333333,63.095238,47.50,69.841270,29.365079,53.928571,1305.0,2.372992e+12,9.322094,1.74382,7.34,0.188501,35.015,0.130,0.032,-0.007605,0.165179,0.441989,0.144,277.0,0.6671,0.279500
8,9,HEXA.JK,Industrials,Barang Perindustrian,Mesin Konstruksi & Kendaraan Berat,52.634921,13.690476,55.634921,87.50,50.793651,28.809524,67.857143,4510.0,3.788400e+12,7.735716,1.34452,9.47,0.147549,93.906,0.348,0.210,0.022676,-0.032189,0.032037,0.085,286.0,0.7613,0.185980
9,10,BFIN.JK,Financial Services,Jasa Pembiayaan,Pembiayaan Konsumen,51.674603,57.142857,71.428571,30.00,63.492063,23.095238,68.452381,820.0,1.217649e+13,7.961165,1.15760,8.43,0.148484,127.618,0.037,-0.078,0.131034,0.064935,0.051282,0.529,2962.0,0.6505,0.456760


## 6) Generate Sector-Relative Rankings

This optional section ranks stocks within each sector to reduce concentration effects and make peer comparisons more meaningful.


In [24]:
# count how many ranked stocks are in each sector
ranked_stocks['Sector'].value_counts()

Sector
Industrials           6
Consumer Defensive    5
Energy                2
Healthcare            2
Financial Services    2
Basic Materials       2
Technology            1
Consumer Cyclical     1
Name: count, dtype: int64

In [25]:
# count how many ranked stocks are in each sector and sub-industry
ranked_stocks.groupby(['Sector', 'NewSubIndustryName']).size().sort_values(ascending=False)

Sector              NewSubIndustryName                   
Industrials         Operator Pelabuhan                       2
                    Mesin Konstruksi & Kendaraan Berat       2
Basic Materials     Barang Kimia Khusus                      1
Financial Services  Manajemen Investasi                      1
Industrials         Produk & Perlengkapan Bangunan           1
                    Konstruksi Bangunan                      1
Healthcare          Mesin & Komponen Perindustrian           1
                    Farmasi                                  1
Financial Services  Pembiayaan Konsumen                      1
Energy              Produksi Batu Bara                       1
Basic Materials     Mesin Konstruksi & Kendaraan Berat       1
Energy              Penyimpanan & Distribusi Minyak & Gas    1
Consumer Defensive  Ritel & Distributor Makanan              1
                    Produk Susu Olahan                       1
                    Produk Perawatan Tubuh                  

In [26]:
for sector, sector_df in ranked_stocks.groupby('Sector', dropna=False):
    print(f"\n=== Sector: {sector} ({len(sector_df)} rows) ===")
    display(sector_df.reset_index(drop=True))



=== Sector: Basic Materials (2 rows) ===


,Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Current Price,Market Cap,PE Ratio,Pbr,Dividend Yield,Roe,Debt to Equity,Revenue Growth,EPS Growth,ThreeMonth,SixMonth,OneYear,Beta,Frequency,Payout Ratio,Free Float Ratio
0,16,AVIA.JK,Basic Materials,Barang Baku,Barang Kimia Khusus,45.174603,52.380952,25.793651,55.0,19.841270,72.539683,31.071429,390.0,2.313196e+13,13.402061,2.53263,6.12,0.183168,0.904,0.063,0.107,-0.210526,-0.080189,-0.029851,0.054,4151.0,0.7558,0.16781
1,17,UNTR.JK,Basic Materials,Barang Perindustrian,Mesin Konstruksi & Kendaraan Berat,44.549603,25.595238,88.015873,15.0,82.539683,66.428571,52.142857,31975.0,1.142380e+14,7.831789,1.15644,6.49,0.143601,18.342,-0.116,-0.153,0.175551,0.186456,0.396288,-0.170,5778.0,0.5024,0.37854



=== Sector: Consumer Cyclical (1 rows) ===


,Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Current Price,Market Cap,PE Ratio,Pbr,Dividend Yield,Roe,Debt to Equity,Revenue Growth,EPS Growth,ThreeMonth,SixMonth,OneYear,Beta,Frequency,Payout Ratio,Free Float Ratio
0,7,SMSM.JK,Consumer Cyclical,Otomotif & Komponen Otomotif,Suku Cadang Otomotif,55.746032,72.619048,53.412698,40.0,50.793651,53.253968,58.571429,1785.0,1.027924e+13,9.157603,2.38675,7.84,0.261147,4.017,0.054,0.01,0.017094,-0.01108,0.031792,0.131,904.0,0.7179,0.41435



=== Sector: Consumer Defensive (5 rows) ===


...

,Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Current Price,Market Cap,PE Ratio,Pbr,Dividend Yield,Roe,Debt to Equity,Revenue Growth,EPS Growth,ThreeMonth,SixMonth,OneYear,Beta,Frequency,Payout Ratio,Free Float Ratio
0,2,MARK.JK,Healthcare,Barang Perindustrian,Mesin & Komponen Perindustrian,63.897817,93.452381,34.126984,71.25,49.206349,28.809524,61.309524,800.0,3.040000e+12,10.706638,3.36792,11.25,0.314501,7.307,0.145,0.219,-0.036145,0.19403,-0.064327,0.295,357.0,0.9371,0.19732
1,12,SIDO.JK,Healthcare,Farmasi & Riset Kesehatan,Farmasi,51.061508,92.857143,24.444444,42.50,22.222222,76.349206,24.047619,505.0,1.486478e+13,12.209865,4.85509,7.33,0.393920,0.132,0.045,0.045,-0.064815,-0.09009,-0.098214,0.106,5315.0,1.0397,0.20860



=== Sector: Industrials (6 rows) ===


,Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Current Price,Market Cap,PE Ratio,Pbr,Dividend Yield,Roe,Debt to Equity,Revenue Growth,EPS Growth,ThreeMonth,SixMonth,OneYear,Beta,Frequency,Payout Ratio,Free Float Ratio
0,3,TOTL.JK,Industrials,Konstruksi Bangunan,Konstruksi Bangunan,57.208333,31.547619,42.142857,90.00,93.650794,54.365079,51.904762,1220.0,4.160200e+12,10.039500,3.12802,6.70,0.311516,0.367,0.286,0.399,0.161905,0.378531,1.033333,0.241,2014.0,0.6173,0.30258
1,8,IPCC.JK,Industrials,Infrastruktur Transportasi,Operator Pelabuhan,53.136905,58.333333,63.095238,47.50,69.841270,29.365079,53.928571,1305.0,2.372992e+12,9.322094,1.74382,7.34,0.188501,35.015,0.130,0.032,-0.007605,0.165179,0.441989,0.144,277.0,0.6671,0.27950
2,9,HEXA.JK,Industrials,Barang Perindustrian,Mesin Konstruksi & Kendaraan Berat,52.634921,13.690476,55.634921,87.50,50.793651,28.809524,67.857143,4510.0,3.788400e+12,7.735716,1.34452,9.47,0.147549,93.906,0.348,0.210,0.022676,-0.032189,0.032037,0.085,286.0,0.7613,0.18598
3,13,IPCM.JK,Industrials,Infrastruktur Transportasi,Operator Pelabuhan,50.766865,33.928571,77.539683,51.25,57.142857,46.507937,51.666667,332.0,1.754557e+12,8.922333,1.29031,7.28,0.144461,5.370,-0.043,0.219,-0.029240,0.070968,0.229630,-0.094,197.0,0.6534,0.22954
4,15,ARNA.JK,Industrials,Barang Perindustrian,Produk & Perlengkapan Bangunan,47.408730,48.809524,63.174603,30.00,29.365079,51.428571,58.095238,505.0,3.550138e+12,9.025916,1.91630,8.91,0.206999,3.771,0.042,-0.107,-0.056075,-0.028846,-0.165289,0.162,1563.0,0.7688,0.45501
5,21,SKRN.JK,Industrials,Barang Perindustrian,Mesin Konstruksi & Kendaraan Berat,31.787698,48.214286,8.253968,5.00,54.761905,42.460317,51.309524,550.0,3.914900e+12,19.210619,6.58656,17.28,0.325294,43.512,-0.332,-0.419,0.089109,-0.153846,0.527778,-0.050,879.0,1.3976,0.12461



=== Sector: Technology (1 rows) ===


,Rank,Ticker,Sector,NewSubSectorName,NewSubIndustryName,Total Score,Quality Score,Valuation Score,Growth Score,Momentum Score,Risk Score,Income Score,Current Price,Market Cap,PE Ratio,Pbr,Dividend Yield,Roe,Debt to Equity,Revenue Growth,EPS Growth,ThreeMonth,SixMonth,OneYear,Beta,Frequency,Payout Ratio,Free Float Ratio
0,5,MSTI.JK,Technology,Perangkat Lunak & Jasa TI,Jasa & Konsultan TI,56.365079,42.857143,68.095238,60.0,36.507937,60.15873,62.97619,1425.0,4.473668e+12,8.14565,1.95233,9.44,0.239622,4.343,0.151,0.094,0.010638,-0.106583,0.010638,-0.374,504.0,0.7662,0.14997
